In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv


load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
from typing import TypedDict, Annotated, Literal
import operator
from langchain_core.messages import BaseMessage


# --- 1. STATE ---
class BlogState(TypedDict):
    topic: str
    content: str
    evaluation: str  # Click Through Rate

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


# --- 2. NODES ---
def write_blog_node(state: BlogState) -> dict:
    """Ask the LLM to classify the user's message"""
    blog_topic = state["topic"]
    user_msg: str = f"""
        Write a detailed blog post on the topic: '{blog_topic}'.
        Include a catchy title, introduction, subheadings, and a conclusion.
        Keep it engaging, informative, and easy to read.
    """
    resp = llm.invoke(
        [
            SystemMessage("You are an expert blog writer. Write a high-quality, engaging, SEO-optimized blog post."),
            HumanMessage(user_msg),
        ]
    )
    blog_content = resp.content.strip()
    print(f"[blog_content] → '{blog_content}'")
    return {"content": blog_content}


def evalute_node(state: BlogState) -> dict:
    """Ask the llm to evalute the blog"""
    blog_content = state["content"]
    resp = llm.invoke(
        [
            SystemMessage(
                "You are a strict blog evaluator and SEO expert. "
                "Evaluate content based on clarity, engagement, structure, SEO, and usefulness. "
                "Be critical and specific."
            ),
            HumanMessage(
                f"Evaluate the following blog post:\n\n{blog_content}\n\n"
                "Return your evaluation in this JSON format:\n"
                "{\n"
                '  "overall_score": number (0-10),\n'
                '  "ctr_score": number (0-10),\n'
                '  "seo_score": number (0-10),\n'
                '  "readability_score": number (0-10),\n'
                '  "strengths": [list of key strengths],\n'
                '  "weaknesses": [list of key issues],\n'
                '  "improvement_suggestions": [list of actionable fixes]\n'
                "}\n\n"
                "Scoring guidelines:\n"
                "- CTR: Title appeal, hook strength, curiosity\n"
                "- SEO: keyword usage, headings, structure\n"
                "- Readability: clarity, flow, simplicity\n"
                "- Overall: combined quality and usefulness\n"
            ),
        ]
    )

    return {"evaluation": resp.content}